In [148]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import accuracy_score

In [149]:
pd.set_option('display.max_columns', None)

In [150]:
test = pd.read_csv("titanic/test.csv")
train = pd.read_csv("titanic/train.csv")

In [151]:
passengers = pd.DataFrame(test['PassengerId'])
train = train.set_index('PassengerId')
test = test.set_index('PassengerId')
train.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [152]:
missing_values = (train.isnull().sum())/len(train)
missing_values.sort_values()

Survived    0.000000
Pclass      0.000000
Name        0.000000
Sex         0.000000
SibSp       0.000000
Parch       0.000000
Ticket      0.000000
Fare        0.000000
Embarked    0.002245
Age         0.198653
Cabin       0.771044
dtype: float64

In [ ]:
## drop Cabin because mostly empty
train = train.drop(columns=['Cabin'])

## drop port of embarkation because shouldn't affect survival - intuitively the port where the passenger embarked is no longer relevant once on the ship
train = train.drop(columns=['Embarked'])

## drop identified data - could lead to overfitting
train = train.drop(columns=['Name','Ticket'])

## apply same to test
test = test.drop(columns=['Cabin','Embarked','Name','Ticket'])

In [155]:
## replace missing ages with mean
train['Age'] = train['Age'].fillna(train['Age'].mean())
test['Age'] = test['Age'].fillna(test['Age'].mean())

In [156]:
y_train = train[['Survived']].to_numpy()
X_train = train.drop(columns=['Survived'])
# X_train = X_train.apply(pd.to_numeric)
# X_train['one'] = 1

In [157]:
## one hot encode Sex
X_train = pd.get_dummies(X_train, columns=["Sex"],dtype=int)
X_test = pd.get_dummies(test, columns=["Sex"],dtype=int)

In [158]:
naInTest = X_test.isna().sum()

In [159]:
X_test = X_test.fillna(0)

In [160]:
y_train = y_train.flatten()

In [161]:
lr = LogisticRegression(random_state=0).fit(X_train, y_train)
y_pred = lr.predict(X_test)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [162]:
y_pred = pd.DataFrame(y_pred)
y_pred.columns = ["Survived"]

In [163]:
y_pred = passengers.join(y_pred)

In [164]:
y_pred

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [165]:
y_pred.to_csv('y_pred.csv', index=False)
